In [76]:
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import pandas as pd
import numpy as np
from fredapi import Fred

In [ ]:
import os
start_date = datetime(year=2010, month=1, day=1)
end_date = datetime(year=2025, month=12, day=31)

RAW_PATH = "../data/raw/"
PROC_PATH = "../data/processed/"

FRED_KEY = os.environ["FRED_API_KEY"]  # free key: https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
# S&P 500 daily

sp500 = yf.download(
    tickers = "^SPX",
    start = start_date,
    end = end_date,
    interval = "1d",
    auto_adjust = True,
    progress = False
)

sp500.to_csv(RAW_PATH + "sp500_raw.csv")
sp500.tail(3)

Price,Close,High,Low,Open,Volume
Ticker,^SPX,^SPX,^SPX,^SPX,^SPX
Date,,,,,
2025-12-26,6929.939941,6945.770020,6921.600098,6936.020020,2586550000
2025-12-29,6905.740234,6920.209961,6888.759766,6903.600098,3541750000
2025-12-30,6896.240234,6913.250000,6893.470215,6900.439941,3309930000


In [79]:
sp500.columns

MultiIndex([( 'Close', '^SPX'),
            (  'High', '^SPX'),
            (   'Low', '^SPX'),
            (  'Open', '^SPX'),
            ('Volume', '^SPX')],
           names=['Price', 'Ticker'])

In [80]:
# Log-returns

close = sp500[("Close", "^SPX" )]
log_returns = np.log(close / close.shift(1)).dropna()
log_returns.name = "log_return"

log_returns.head()

Date
2010-01-05    0.003111
2010-01-06    0.000545
2010-01-07    0.003993
2010-01-08    0.002878
2010-01-11    0.001745
Name: log_return, dtype: float64

In [81]:
# Treasury yields — 10Y and 2Y

tnx = yf.download("^TNX", start=start_date, end=end_date, interval="1d", auto_adjust=True, progress=False)[("Close", "^TNX")].rename("Yield_10Y")
irx = yf.download("^IRX", start=start_date, end=end_date, interval="1d", auto_adjust=True, progress=False)[("Close", "^IRX")].rename("Yield_2Y")

yields = pd.concat([tnx, irx], axis=1)
yields["Spread_2Y10Y"] = yields["Yield_2Y"] - yields["Yield_10Y"]

yields.to_csv(RAW_PATH + "treasury_yields_raw.csv")
yields.tail(3)

,Yield_10Y,Yield_2Y,Spread_2Y10Y
Date,,,
2025-12-26,4.136,3.543,-0.593
2025-12-29,4.116,3.538,-0.578
2025-12-30,4.130,3.540,-0.590


In [82]:
# VIX

vix = yf.download("^VIX", start=start_date, end=end_date, interval="1d", auto_adjust=True, progress=False)[("Close", "^VIX")].rename("VIX")

vix.to_csv(RAW_PATH + "vix_raw.csv")
vix.tail(3)

Date
2025-12-26    13.60
2025-12-29    14.20
2025-12-30    14.33
Name: VIX, dtype: float64

In [ ]:
import os
# FRED — EFFR, CPI, Unemployment

fred = Fred(api_key=FRED_KEY)

effr = fred.get_series("EFFR", observation_start=start_date, observation_end=end_date).rename("EFFR")
cpi = fred.get_series("CPIAUCSL", observation_start=start_date, observation_end=end_date).rename("CPI")
unemp = fred.get_series("UNRATE", observation_start=start_date, observation_end=end_date).rename("Unemployment")

cpi_yoy = cpi.pct_change(12) * 100

df_fred = pd.DataFrame({"EFFR": effr, "CPI_YoY": cpi_yoy, "Unemployment": unemp})
df_fred = df_fred.ffill()
df_fred.to_csv(RAW_PATH + "fred_macro_raw.csv")
df_fred.tail(3)

<path> FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  cpi_yoy = cpi.pct_change(12) * 100


,EFFR,CPI_YoY,Unemployment
2025-12-29,3.64,2.653304,4.4
2025-12-30,3.64,2.653304,4.4
2025-12-31,3.64,2.653304,4.4


In [84]:
# NFCI — weekly

nfci = pd.read_csv(RAW_PATH + "nfci-data-series-csv.csv")
nfci["Date"] = pd.to_datetime(nfci["Friday_of_Week"])
nfci = nfci.set_index("Date")[["NFCI"]].sort_index()
nfci = nfci.loc["2010-01-01":"2025-12-31"]

nfci.tail(3)

,NFCI
Date,
2025-12-12,-0.518
2025-12-19,-0.526
2025-12-26,-0.537


In [85]:
# EPU — monthly

epu = pd.read_csv(RAW_PATH + "US_epu.csv", sep=";")
epu["Date"] = pd.to_datetime({"year": epu["Year"], "month": epu["Month"], "day": 1})
epu = epu.set_index("Date")[["News_Based_Policy_Uncert_Index"]].rename(columns={"News_Based_Policy_Uncert_Index": "EPU"}).sort_index()
epu = epu.loc["2010-01-01":"2025-12-31"]

epu.tail(3)

,EPU
Date,
2025-10-01,308
2025-11-01,312
2025-12-01,289


In [86]:
# GPR — monthly

gpr = pd.read_excel(RAW_PATH + "data_gpr_export.xls")
gpr["Date"] = pd.to_datetime(gpr["month"])
gpr = gpr.set_index("Date")[["GPR"]].sort_index()
gpr = gpr.loc["2010-01-01":"2025-12-31"]

gpr.tail(3)

,GPR
Date,
2025-10-01,154.512924
2025-11-01,104.338654
2025-12-01,130.955856


In [90]:
# Merge — forward-fill low-frequency series to daily

epu.index.name = "Date"
gpr.index.name = "Date"

df = log_returns.to_frame()
df = df.join(nfci, how="left")
df = df.join(epu, how="left")
df = df.join(gpr, how="left")
df = df.join(yields, how="left")
df = df.join(vix, how="left")
df = df.join(df_fred[["EFFR", "CPI_YoY", "Unemployment"]].resample("D").ffill(), how="left")
df = df.loc[:, ~df.columns.duplicated()]

df[["NFCI", "EPU", "GPR", "EFFR", "CPI_YoY"]] = df[["NFCI", "EPU", "GPR", "EFFR", "CPI_YoY"]].ffill().bfill()

df.to_csv(PROC_PATH + "merged_data.csv")

print(df.shape)
print(df.isnull().sum())
df.tail(5)

(4022, 11)
log_return      0
NFCI            0
EPU             0
GPR             0
Yield_10Y       2
Yield_2Y        2
Spread_2Y10Y    2
VIX             0
EFFR            0
CPI_YoY         0
Unemployment    0
dtype: int64


,log_return,NFCI,EPU,GPR,Yield_10Y,Yield_2Y,Spread_2Y10Y,VIX,EFFR,CPI_YoY,Unemployment
Date,,,,,,,,,,,
2025-12-23,0.004540,-0.526,289.0,130.955856,4.169,3.547,-0.622,14.00,3.64,2.653304,4.4
2025-12-24,0.003216,-0.526,289.0,130.955856,4.136,3.555,-0.581,13.47,3.64,2.653304,4.4
2025-12-26,-0.000304,-0.537,289.0,130.955856,4.136,3.543,-0.593,13.60,3.64,2.653304,4.4
2025-12-29,-0.003498,-0.537,289.0,130.955856,4.116,3.538,-0.578,14.20,3.64,2.653304,4.4
2025-12-30,-0.001377,-0.537,289.0,130.955856,4.130,3.540,-0.590,14.33,3.64,2.653304,4.4


In [92]:
# Pour l'utlisation mfGARCH

df[["log_return", "NFCI", "EPU", "GPR"]].to_csv("../data/processed/data_for_R.csv")